[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [3]:
import torch

In [4]:

# ✏️ YOUR IMPLEMENTATION HERE

def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
   _max = torch.max(x, dim=dim, keepdim=True).values
   x = x - _max
   numerator = torch.exp(x)
   denominator = torch.sum(numerator, dim=dim, keepdim=True)
   return numerator / denominator

In [5]:
def relu(x: torch.Tensor) -> torch.Tensor:
    return torch.where(x > 0, x, torch.zeros_like(x))

In [33]:
def my_log_softmax_trap(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
  """
  The bug is in my_log_softmax. Computing softmax first and then taking the log is the trap I mentioned earlier. Here is the exact chain that produces the NaN.

  How the NaN happens

  Take one row with large logits, like [0, 1000].

  After subtracting the max: [-1000, 0]
  exp: [e^-1000, 1], and e^-1000 underflows to exactly 0.0 in float32
  softmax: [0.0, 1.0]
  log: [-inf, 0.0]
  """
  return torch.log(my_softmax(logits, dim=dim))

## Why `log(softmax(x))` gives NaN

The bug is in `my_log_softmax`: computing softmax first and then taking the log.

### How the NaN happens

Take one row with large logits: `[0, 1000]`.

1. Subtract the max: `[-1000, 0]`
2. `exp`: `[e^-1000, 1]`. In float32, `e^-1000` **underflows to exactly `0.0`**.
3. softmax: `[0.0, 1.0]`
4. `log`: `[-inf, 0.0]`

The true log-softmax value at index 0 is about `-1000`, a normal number. It was lost at step 2, when the probability rounded to 0.

### The one-hot step makes it worse

Say the target is class 1:

```
one_hot   = [0,        1  ]
log_probs = [-inf,     0.0]
product   = [0 * -inf, 0.0] = [NaN, 0.0]
```

In floating point, `0 * -inf = NaN`, not 0. A **wrong** class poisons the sum, even though the correct class's value was fine.

> Masking with a one-hot multiply is safe in math, but not in floating point when infinities are possible.

---

## The fix: never leave log-space

Let $z = x - \max(x)$. Then:

$$\log \frac{e^{z_i}}{\sum_j e^{z_j}} = \log e^{z_i} - \log \sum_j e^{z_j} = z_i - \log \sum_j e^{z_j}$$

- **$z_i$**: just the shifted logit. No `exp`, no `log`, so nothing can underflow. You get `-1000` exactly.
- **$\log \sum_j e^{z_j}$**: after the max shift, the largest term is $e^0 = 1$, so the sum is at least 1 and its log is finite.

So `my_log_softmax` should **not** call `my_softmax`. It should:

1. take the max along `dim` (with `keepdim=True`)
2. subtract it
3. subtract the log of the summed exps (with `keepdim=True` again)

This is the same as the cross-entropy form:

$$\text{CE} = -x_y + \log \sum_j e^{x_j}$$

---

## Picking the correct class (secondary)

Once log-softmax has no `-inf`, the one-hot version works. A more direct way: select one value per row. For row `i`, take column `targets[i]`.

- Indexing with two index tensors: `x[rows, cols]`
- or `torch.gather`

This reads exactly like $x_y$ in the formula and never multiplies by zeros.

---

## Lesson

`log(softmax(x))` and `log_softmax(x)` are equal in math, but only the second is safe to compute. This is why `F.cross_entropy` takes **logits**, not probabilities.

In [56]:
def my_log_softmax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
  r"""
  logits: (B, C) float, targets: (B,) long indices
  Returns: (B, C) float

  Let $z = x - \max(x)$. Then:

  $$\log \frac{e^{z_i}}{\sum_j e^{z_j}} = \log e^{z_i} - \log \sum_j e^{z_j} = z_i - \log \sum_j e^{z_j}$$
  """
  # shift to normalize
  z = logits - torch.max(logits, dim=dim, keepdim=True).values
  answer = z - torch.log(torch.sum(torch.exp(z), dim=dim, keepdim=True))
  return answer

In [49]:
def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor):
  # logits: (B, C) float, targets: (B,) long indices
  # logits shape: 4, 10, targets shape: (4,)
  # Returns: scalar loss (mean over batch)

  # 1. Calculate log_probabilities using my_log_softmax for numerical stability.
  log_predicted_propability = my_log_softmax(logits=logits, dim=1)

  # 2. Convert targets to one-hot encoding.
  num_classes = logits.shape[1]
  targets_one_hot = torch.nn.functional.one_hot(targets, num_classes=num_classes).float()
  return torch.mean(-1 * torch.sum(targets_one_hot * log_predicted_propability, dim=1))

In [51]:
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))

In [52]:
logits

tensor([[ 0.1076, -1.0715, -0.1166, -1.0170, -1.4453, -0.8078,  1.1975, -1.8345,
          1.5435, -0.0332],
        [-0.4186, -0.2556, -0.3882, -0.3342,  0.9523, -0.4624,  1.9351,  1.0077,
          1.0046, -0.4335],
        [ 1.1685,  0.7704,  0.3907,  0.2896, -0.3099, -0.0561,  0.5174, -1.5962,
          0.5411,  1.5390],
        [ 1.0860,  1.2464,  0.5631, -1.4351,  0.7194, -1.3707,  0.8732,  0.0651,
          0.7732, -0.9701]])

In [53]:
targets

tensor([6, 0, 0, 4])

In [54]:
# 🧪 Debug

print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

Loss: tensor(2.1312)
Loss: tensor(2.1312)
Ref:  tensor(2.1312)


In [55]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Matches F.cross_entropy (5.9ms)
  ✅ [2/4] Numerical stability (0.7ms)
  ✅ [3/4] Scalar output (0.4ms)
  ✅ [4/4] Gradient flow (1.5ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.5ms total)
  Progress saved. Run status() to see your dashboard.

